# 🎂 Our Friendship in Data
A birthday surprise built from a WhatsApp chat.

**How to use**
1. Export the chat in WhatsApp (⋮ → More → Export chat → *Without media*) and save it as `chat.txt` **in the same folder as this notebook**.
2. Change the settings in the cell below.
3. Click **Run → Run All Cells**.
4. Your pictures are saved in the `surprise_images` folder. Send her your favourites. 💖

🔒 Everything runs on your own computer. Your chat is never uploaded.

## Step 1: Install the tools (run once)

In [ ]:
%pip install pandas matplotlib wordcloud -q

## Step 2: Your settings ✏️

In [ ]:
HER_NAME = "Priya"
YOUR_NAME = "Ravi"
BIRTHDAY_MESSAGE = ("Thank you for every late-night chat, every silly joke and every time you "
                    "showed up for me. Happy Birthday to the best friend ever! 💖")
CHAT_FILE = "chat.txt"

## Step 3: Load the chat into a table

In [ ]:
import os, re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("surprise_images", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
PINK, PURPLE = "#d6336c", "#7048e8"

LINE = re.compile(r"^\[?(\d{1,2})/(\d{1,2})/(\d{2,4}),?\s+(\d{1,2}):(\d{2})(?::\d{2})?\s?([apAP][mM])?\]?\s*-?\s*([^:]+?):\s(.*)$")

rows = []
with open(CHAT_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.replace("\u200e", "").replace("\u202f", " ").rstrip("\n")
        m = LINE.match(line)
        if m:
            rows.append(list(m.groups()))
        elif rows:                       # a multi-line message continues here
            rows[-1][-1] += " " + line

df = pd.DataFrame(rows, columns=["a", "b", "y", "hour", "minute", "ampm", "sender", "text"])
df[["a", "b", "y", "hour", "minute"]] = df[["a", "b", "y", "hour", "minute"]].astype(int)

# Is the date day/month or month/day? Decide from the data.
month_first = (df.b > 12).any() and not (df.a > 12).any()
df["day"], df["month"] = (df.b, df.a) if month_first else (df.a, df.b)
df["year"] = df.y.where(df.y > 99, df.y + 2000)

pm = df.ampm.fillna("").str.lower()
has_ampm = pm != ""
df.loc[has_ampm, "hour"] = (df.hour % 12 + (pm == "pm") * 12)[has_ampm]

df["date"] = pd.to_datetime(df[["year", "month", "day"]])
df["weekday"] = df.date.dt.day_name()
df["sender"] = df.sender.str.strip()
df = df[["date", "weekday", "hour", "sender", "text"]]
print(f"Loaded {len(df):,} messages")
df.head()

## Step 4: The big numbers 🔢

In [ ]:
days_known = (df.date.max() - df.date.min()).days
per_person = df.sender.value_counts().head(2)

print(f"💬 {len(df):,} messages in {days_known:,} days")
print(f"📅 First message: {df.date.min():%d %B %Y}")

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(per_person.index[::-1], per_person.values[::-1], color=[PURPLE, PINK])
for i, v in enumerate(per_person.values[::-1]):
    ax.text(v, i, f"  {v:,}", va="center")
ax.set_title(f"Who talks more? ({len(df):,} messages in total)")
ax.set_xlim(0, per_person.max() * 1.15)
plt.tight_layout(); plt.savefig("surprise_images/1_messages.png"); plt.show()

## Step 5: Our favourite words ☁️

In [ ]:
STOP = set("""the and for with you are was this that not have has had its dont just what when how why who her his she
they them from all get got lol haha media omitted message deleted null yes okay hai haan nahi kya kar toh bhi""".split())

words = Counter(w for t in df.text if "omitted" not in t.lower()
                for w in re.findall(r"[a-z']{3,}", t.lower()) if w not in STOP)

try:
    from wordcloud import WordCloud
    wc = WordCloud(width=1000, height=500, background_color="white", colormap="cool",
                   max_words=60).generate_from_frequencies(words)
    plt.figure(figsize=(9, 4.5)); plt.imshow(wc); plt.axis("off")
    plt.title("The words that are 'us'")
    plt.tight_layout(); plt.savefig("surprise_images/2_words.png"); plt.show()
except Exception as e:                    # fallback if wordcloud isn't installed
    top = pd.Series(dict(words.most_common(15)))
    top[::-1].plot.barh(color=PURPLE, figsize=(7, 5), title="Our favourite words")
    plt.tight_layout(); plt.savefig("surprise_images/2_words.png"); plt.show()

print("Top 10:", ", ".join(f"{w} ({c})" for w, c in words.most_common(10)))

## Step 6: Our emojis 😍

In [ ]:
emoji_re = re.compile(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]")
emojis = Counter(e for t in df.text for e in emoji_re.findall(t))
pd.DataFrame(emojis.most_common(8), columns=["emoji", "times used"])

## Step 7: When do we talk? 🕒

In [ ]:
by_hour = df.hour.value_counts().reindex(range(24), fill_value=0)
peak = by_hour.idxmax()
night = by_hour[[23, 0, 1, 2, 3, 4]].sum()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(by_hour.index, by_hour.values, color=[PINK if h == peak else "#b197fc" for h in by_hour.index])
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f"{h%12 or 12} {'AM' if h < 12 else 'PM'}" for h in range(0, 24, 3)])
ax.set_title(f"Our favourite time to chat: {peak%12 or 12} {'AM' if peak < 12 else 'PM'}")
plt.tight_layout(); plt.savefig("surprise_images/3_hours.png"); plt.show()
print(f"🦉 {night:,} messages were sent between 11 PM and 5 AM")

### Our week, as a heatmap 🔥

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
grid = (df.groupby(["weekday", "hour"]).size().unstack(fill_value=0)
          .reindex(index=order, columns=range(24), fill_value=0))

plt.figure(figsize=(10, 3.6))
plt.imshow(grid, aspect="auto", cmap="RdPu")
plt.yticks(range(7), order); plt.xticks(range(0, 24, 3), [f"{h}:00" for h in range(0, 24, 3)])
plt.colorbar(label="messages"); plt.title("When we talk (day x hour)")
plt.tight_layout(); plt.savefig("surprise_images/4_heatmap.png"); plt.show()

## Step 8: Our friendship over time 📈

In [ ]:
monthly = df.groupby(df.date.dt.to_period("M")).size()
best_day = df.date.value_counts().idxmax()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(monthly.index.astype(str), monthly.values, color=PINK, marker="o", linewidth=2)
ax.fill_between(range(len(monthly)), monthly.values, color=PINK, alpha=.15)
ax.set_xticks(range(0, len(monthly), max(1, len(monthly)//8)))
ax.set_xticklabels(monthly.index.astype(str)[::max(1, len(monthly)//8)], rotation=30)
ax.set_title("Messages per month")
plt.tight_layout(); plt.savefig("surprise_images/5_timeline.png"); plt.show()

print(f"🏆 Our biggest chat day: {best_day:%d %B %Y} with {df.date.value_counts().max():,} messages. What happened that day? 😄")

## Step 9: The birthday card 🎁

In [ ]:
from IPython.display import HTML
HTML(f"""
<div style="font-family:Georgia,serif;text-align:center;padding:36px;border-radius:24px;
            background:linear-gradient(160deg,#ffe3ec,#e6e0ff);color:#3a2b4d;max-width:640px;margin:auto">
  <h1 style="margin:0">🎂 Happy Birthday, {HER_NAME}! 🎂</h1>
  <p style="font-size:1.2rem;margin:18px 0 6px"><b>{len(df):,}</b> messages and counting…</p>
  <p style="font-size:1.25rem;font-style:italic;line-height:1.6">{BIRTHDAY_MESSAGE}</p>
  <p style="font-size:1.1rem">With love, <b>{YOUR_NAME}</b> 💖</p>
</div>""")

## Ideas to make it even better
- Add a photo of the two of you with `IPython.display.Image("photo.jpg")`
- Print a poster: open the `surprise_images` folder and print your favourite chart
- Add your own inside-joke words to the `STOP` list or the word cloud